# 🐼 Pandas pour auditeurs — de Excel au code

**Objectif** : découvrir et *expérimenter* les fonctions les plus classiques de **pandas**
(la bibliothèque Python de manipulation de données) en partant de ce que vous connaissez
déjà dans Excel.

**Public** : auditeurs bancaires, à l'aise avec Excel (sommes automatiques, `RECHERCHEV` / `VLOOKUP`),
mais **non spécialistes de Python**.

**Comment lire ce notebook**
- Chaque cellule grise est du *code*. Cliquez dedans puis appuyez sur **`Maj` + `Entrée`** pour l'exécuter.
- **Exécutez les cellules dans l'ordre, de haut en bas.** Une cellule peut dépendre des précédentes.
- À chaque notion, un encadré vous rappelle l'**équivalent Excel** :

> 💡 **Équivalent Excel** : à quoi cela correspond dans un tableur.

- Des sections **🛠️ À vous de jouer** vous proposent un exercice. La **✅ Solution** suit juste après
  (essayez d'abord par vous-même !).

> ℹ️ **Aucune donnée réelle** n'est utilisée. Les données de transactions sont *générées aléatoirement*
> dans ce notebook ; vous pouvez donc tout exécuter et tout casser sans risque.


## Sommaire

1. Mise en route & vocabulaire (`DataFrame`)
2. Générer un jeu de transactions de démonstration
3. Lire & écrire des fichiers (`read_csv`, `read_excel`)
4. Regarder ses données (`head`, `info`, `describe`)
5. Sélectionner colonnes et lignes (`loc`, `iloc`)
6. Filtrer (l'équivalent des *filtres* Excel)
7. Trier (`sort_values`)
8. Créer / calculer des colonnes
9. Agréger : `sum`, `mean`, `count` (≈ `SOMME`, `MOYENNE`, `NB`)
10. Grouper : `groupby` (≈ `SOMME.SI` / `SUMIF`)
11. Tableaux croisés : `pivot_table` (≈ *tableau croisé dynamique*)
12. Rapprocher deux tables : `merge` (≈ `RECHERCHEV` / `VLOOKUP`)
13. Travailler avec les dates
14. Qualité des données : valeurs manquantes & doublons
15. 🔎 Cas d'audit concrets (mini-revue analytique)
16. Exporter ses résultats


## 1. Mise en route & vocabulaire

On *importe* d'abord les deux outils dont on a besoin. C'est l'équivalent d'« ouvrir Excel »
avant de commencer.

- `pandas` : pour manipuler des tableaux de données. On lui donne traditionnellement le surnom `pd`.
- `numpy` : pour générer des nombres (on s'en sert seulement pour fabriquer le jeu de démo). Surnom `np`.

> 💡 **Équivalent Excel** : lancer Excel. Ici, on charge les fonctionnalités une seule fois.


In [1]:
import pandas as pd
import numpy as np

# Afficher toutes les colonnes sans les tronquer
pd.set_option("display.max_columns", None)

print("pandas version :", pd.__version__)
print("Tout est prêt ✅")

pandas version : 3.0.1
Tout est prêt ✅


### Le mot le plus important : `DataFrame`

Un **`DataFrame`** est tout simplement **un tableau** : des **colonnes** (avec un nom) et des **lignes**
(numérotées à partir de 0). C'est l'équivalent exact d'**une feuille Excel** ou d'un **tableau structuré**.

| Vocabulaire pandas | Équivalent Excel |
|---|---|
| `DataFrame` | une feuille / un tableau |
| une `column` (colonne) | une colonne (A, B, C…) mais nommée |
| une `row` (ligne) | une ligne |
| l'`index` | le numéro de ligne (commence à **0**) |
| une `Series` | une seule colonne isolée |

Petit exemple jouet pour visualiser :


In [2]:
exemple = pd.DataFrame({
    "compte":   ["A001", "A002", "A003"],
    "montant":  [1500, 320, 9800],
    "devise":   ["EUR", "EUR", "USD"],
})

exemple

,compte,montant,devise
0,A001,1500,EUR
1,A002,320,EUR
2,A003,9800,USD


Remarquez la colonne tout à gauche (0, 1, 2) : c'est l'**index**, l'équivalent du numéro de ligne.
⚠️ En Python, **on compte à partir de 0**, pas de 1.

## 2. Générer un jeu de transactions de démonstration

On fabrique ~500 transactions bancaires fictives. **Vous n'avez pas besoin de comprendre ce code** :
exécutez-le simplement. Dans la vraie vie, ces données viendraient d'un export de votre core banking
ou d'un fichier Excel (voir section 3).

Quelques « pièges » réalistes ont été **glissés volontairement** dans les données (doublons,
valeurs manquantes, montants ronds, montants juste sous un seuil…) pour les cas d'audit de la section 15.


In [3]:
np.random.seed(42)   # rend le tirage reproductible : tout le monde obtient les mêmes données
n = 500

dates = pd.to_datetime("2024-01-01") + pd.to_timedelta(np.random.randint(0, 365, n), unit="D")

comptes = ["FR7610011000601234567890185", "FR7630004000031234567890143",
           "FR7612548029981234567890161", "FR7620041010051234567890138"]

contreparties = ["ALPHA SARL", "BETA SA", "GAMMA GMBH", "DELTA LTD",
                 "EPSILON SAS", "ZETA BV", "Particulier", "ETA TRADING"]

df = pd.DataFrame({
    "transaction_id":   range(1, n + 1),
    "date":             dates,
    "account_id":       np.random.choice(comptes, n),
    "counterparty":     np.random.choice(contreparties, n),
    "transaction_type": np.random.choice(["Virement", "Prélèvement", "Carte", "Espèces", "Chèque"],
                                         n, p=[0.40, 0.20, 0.20, 0.10, 0.10]),
    "sens":             np.random.choice(["Débit", "Crédit"], n, p=[0.6, 0.4]),
    "channel":          np.random.choice(["Online", "Agence", "ATM", "Mobile"], n),
    "amount":           np.round(np.random.lognormal(mean=6.5, sigma=1.2, size=n), 2),
    "currency":         np.random.choice(["EUR", "USD", "GBP"], n, p=[0.85, 0.10, 0.05]),
    "country":          np.random.choice(["LU", "FR", "DE", "BE", "US", "GB"], n),
})

# --- pièges volontaires pour les cas d'audit ---
# montants "ronds"
df.loc[np.random.choice(df.index, 15, replace=False), "amount"] = \
    np.random.choice([1000, 5000, 10000, 50000], 15)
# montants juste SOUS le seuil de déclaration de 10 000 (structuring)
df.loc[np.random.choice(df.index, 8, replace=False), "amount"] = \
    np.random.choice([9900.0, 9950.0, 9800.0, 9990.0], 8)
# valeurs manquantes
df.loc[np.random.choice(df.index, 12, replace=False), "counterparty"] = np.nan
df.loc[np.random.choice(df.index, 7,  replace=False), "country"] = np.nan
# doublons (même transaction enregistrée deux fois, avec un id différent)
dups = df.sample(6, random_state=1).copy()
dups["transaction_id"] = range(n + 1, n + 1 + len(dups))
df = pd.concat([df, dups], ignore_index=True)

# on mélange l'ordre des lignes pour faire réaliste
df = df.sample(frac=1, random_state=7).reset_index(drop=True)

print("Jeu de données prêt :", df.shape[0], "lignes,", df.shape[1], "colonnes")

Jeu de données prêt : 506 lignes, 10 colonnes


**Signification des colonnes**

| Colonne | Description |
|---|---|
| `transaction_id` | identifiant unique de l'opération |
| `date` | date de l'opération |
| `account_id` | IBAN du compte de la banque |
| `counterparty` | contrepartie (donneur d'ordre / bénéficiaire) |
| `transaction_type` | type d'opération |
| `sens` | Débit ou Crédit |
| `channel` | canal (Online, Agence, ATM, Mobile) |
| `amount` | montant de l'opération |
| `currency` | devise |
| `country` | pays de la contrepartie |


## 3. Lire & écrire des fichiers

Dans la pratique, vos données arrivent dans un **fichier** (CSV ou Excel). Pour la démo, on enregistre
d'abord notre jeu dans deux fichiers, puis on montre comment les **relire**.

> 💡 **Équivalent Excel** : *Fichier → Enregistrer sous* (écrire) et *Fichier → Ouvrir* (lire).


In [6]:
# Écrire (export) -- index=False pour ne pas écrire la colonne des numéros de ligne
df.to_csv("transactions.csv", index=False)
df.to_excel("transactions.xlsx", index=False)   # nécessite le paquet openpyxl
print("Fichiers transactions.csv et transactions.xlsx créés.")

Fichiers transactions.csv et transactions.xlsx créés.


In [7]:
# Lire (import) un CSV
df_csv = pd.read_csv("transactions.csv")

# Lire un fichier Excel
df_xlsx = pd.read_excel("transactions.xlsx")

print("Lignes lues depuis le CSV   :", len(df_csv))
print("Lignes lues depuis l'Excel  :", len(df_xlsx))

Lignes lues depuis le CSV   : 506
Lignes lues depuis l'Excel  : 506


> ℹ️ Si `read_excel` / `to_excel` renvoie une erreur disant qu'il manque **openpyxl**, exécutez une fois,
> dans une cellule, la commande : `!pip install openpyxl` (le point d'exclamation lance une installation).

Pour vos vrais fichiers, il suffira de remplacer le nom :
`pd.read_excel("C:/Users/moi/Bureau/mon_export.xlsx")`.
Les autres sections continuent avec la variable `df`.


## 4. Regarder ses données

Premier réflexe d'auditeur : **regarder** ce qu'on a sous la main.

> 💡 **Équivalent Excel** : faire défiler les premières lignes, regarder l'en-tête, compter les lignes.


In [8]:
df.head()        # les 5 premières lignes (head = "tête"). df.head(10) pour 10 lignes.

,transaction_id,date,account_id,counterparty,transaction_type,sens,channel,amount,currency,country
0,358,2024-11-18,FR7612548029981234567890161,DELTA LTD,Virement,Crédit,Online,283.87,USD,GB
1,338,2024-02-06,FR7610011000601234567890185,Particulier,Virement,Débit,Agence,1140.92,EUR,LU
2,328,2024-07-10,FR7612548029981234567890161,ZETA BV,Carte,Crédit,Mobile,292.73,EUR,GB
3,14,2024-12-25,FR7610011000601234567890185,Particulier,Espèces,Débit,Mobile,888.46,EUR,US
4,419,2024-09-18,FR7630004000031234567890143,Particulier,Virement,Crédit,Mobile,187.45,EUR,DE


In [9]:
df.tail(3)       # les 3 dernières lignes (tail = "queue")

,transaction_id,date,account_id,counterparty,transaction_type,sens,channel,amount,currency,country
503,26,2024-01-22,FR7612548029981234567890161,DELTA LTD,Virement,Débit,Agence,870.22,EUR,LU
504,197,2024-11-13,FR7630004000031234567890143,BETA SA,Prélèvement,Crédit,ATM,1051.37,EUR,BE
505,176,2024-10-09,FR7610011000601234567890185,DELTA LTD,Virement,Débit,Agence,4178.00,EUR,GB


In [10]:
df.shape         # (nombre de lignes, nombre de colonnes)

(506, 10)

In [11]:
df.columns       # la liste des noms de colonnes

Index(['transaction_id', 'date', 'account_id', 'counterparty',
       'transaction_type', 'sens', 'channel', 'amount', 'currency', 'country'],
      dtype='str')

In [12]:
df.info()        # type de chaque colonne + nombre de valeurs non manquantes

<class 'pandas.DataFrame'>
RangeIndex: 506 entries, 0 to 505
Data columns (total 10 columns):
 #   Column            Non-Null Count  Dtype         
---  ------            --------------  -----         
 0   transaction_id    506 non-null    int64         
 1   date              506 non-null    datetime64[us]
 2   account_id        506 non-null    str           
 3   counterparty      494 non-null    str           
 4   transaction_type  506 non-null    str           
 5   sens              506 non-null    str           
 6   channel           506 non-null    str           
 7   amount            506 non-null    float64       
 8   currency          506 non-null    str           
 9   country           499 non-null    str           
dtypes: datetime64[us](1), float64(1), int64(1), str(7)
memory usage: 70.3 KB


`df.describe()` calcule d'un coup les **statistiques** des colonnes numériques
(nombre, moyenne, écart-type, min, quartiles, max).

> 💡 **Équivalent Excel** : `NB`, `MOYENNE`, `MIN`, `MAX`, `ÉCARTYPE`… le tout en une ligne.


In [13]:
df.describe()

,transaction_id,date,amount
count,506.000000,506,506.000000
mean,253.500000,2024-07-05 12:54:04.268774,2086.860217
min,1.000000,2024-01-01 00:00:00,22.220000
25%,127.250000,2024-04-12 00:00:00,294.020000
50%,253.500000,2024-07-06 00:00:00,681.315000
75%,379.750000,2024-09-30 18:00:00,1656.555000
max,506.000000,2024-12-30 00:00:00,50000.000000
std,146.213884,NaN,5788.763649


## 5. Sélectionner des colonnes et des lignes

### Une ou plusieurs colonnes

> 💡 **Équivalent Excel** : sélectionner la colonne « montant », ou les colonnes « montant » et « devise ».


In [14]:
df["amount"].head()                       # UNE colonne (entre crochets, son nom entre guillemets)

0     283.87
1    1140.92
2     292.73
3     888.46
4     187.45
Name: amount, dtype: float64

In [15]:
df[["amount", "currency"]].head()          # PLUSIEURS colonnes : une liste [ ... ] de noms

,amount,currency
0,283.87,USD
1,1140.92,EUR
2,292.73,EUR
3,888.46,EUR
4,187.45,EUR


### Des lignes précises avec `.loc` et `.iloc`

- `.iloc[...]` : sélection par **position** (i comme *integer*, le numéro). On compte à partir de 0.
- `.loc[...]` : sélection par **étiquette** (label) d'index et **nom** de colonne.

> 💡 **Équivalent Excel** : aller à une cellule/plage précise, par exemple `B2:C5`.


In [16]:
df.iloc[0]              # la toute première ligne (position 0)

transaction_id                              358
date                        2024-11-18 00:00:00
account_id          FR7612548029981234567890161
counterparty                          DELTA LTD
transaction_type                       Virement
sens                                     Crédit
channel                                  Online
amount                                   283.87
currency                                    USD
country                                      GB
Name: 0, dtype: object

In [17]:
df.iloc[0:5]            # les 5 premières lignes (positions 0 à 4)

,transaction_id,date,account_id,counterparty,transaction_type,sens,channel,amount,currency,country
0,358,2024-11-18,FR7612548029981234567890161,DELTA LTD,Virement,Crédit,Online,283.87,USD,GB
1,338,2024-02-06,FR7610011000601234567890185,Particulier,Virement,Débit,Agence,1140.92,EUR,LU
2,328,2024-07-10,FR7612548029981234567890161,ZETA BV,Carte,Crédit,Mobile,292.73,EUR,GB
3,14,2024-12-25,FR7610011000601234567890185,Particulier,Espèces,Débit,Mobile,888.46,EUR,US
4,419,2024-09-18,FR7630004000031234567890143,Particulier,Virement,Crédit,Mobile,187.45,EUR,DE


In [ ]:
# .loc avec un nom de colonne : les colonnes date + montant des 5 premières lignes
df.loc[0:4, ["date", "amount"]]

,date,amount
0,2024-11-18,283.87
1,2024-02-06,1140.92
2,2024-07-10,292.73
3,2024-12-25,888.46
4,2024-09-18,187.45


### 🛠️ À vous de jouer #1

Affichez seulement les colonnes **`transaction_id`**, **`counterparty`** et **`amount`**
pour les **10 premières** transactions.


In [ ]:
# Votre code ici


**✅ Solution**

In [ ]:
df[["transaction_id", "counterparty", "amount"]].head(10)

## 6. Filtrer les lignes (les *filtres* d'Excel)

C'est sans doute l'opération la plus utile en audit : **ne garder que les lignes qui remplissent
une condition**.

Le principe : on écrit une **condition** entre crochets, et pandas ne garde que les lignes où elle est vraie.

> 💡 **Équivalent Excel** : les *filtres automatiques*, ou la fonction `FILTRE()`.


In [19]:
# Toutes les transactions de plus de 10 000
df[df["amount"] > 10000].head()

,transaction_id,date,account_id,counterparty,transaction_type,sens,channel,amount,currency,country
22,203,2024-04-22,FR7610011000601234567890185,EPSILON SAS,Prélèvement,Débit,Online,16543.22,EUR,BE
26,307,2024-06-24,FR7630004000031234567890143,Particulier,Virement,Débit,Mobile,17311.67,EUR,US
72,355,2024-08-10,FR7630004000031234567890143,NaN,Chèque,Crédit,ATM,50000.00,EUR,GB
93,207,2024-01-02,FR7630004000031234567890143,ALPHA SARL,Chèque,Débit,Mobile,50000.00,EUR,US
181,278,2024-02-23,FR7620041010051234567890138,ALPHA SARL,Virement,Crédit,Online,10948.92,EUR,DE


In [20]:
# Toutes les opérations en espèces ( == veut dire "est égal à")
df[df["transaction_type"] == "Espèces"].head()

,transaction_id,date,account_id,counterparty,transaction_type,sens,channel,amount,currency,country
3,14,2024-12-25,FR7610011000601234567890185,Particulier,Espèces,Débit,Mobile,888.46,EUR,US
8,66,2024-04-15,FR7610011000601234567890185,ETA TRADING,Espèces,Débit,Mobile,773.57,EUR,LU
10,389,2024-04-01,FR7630004000031234567890143,ETA TRADING,Espèces,Débit,Agence,274.08,EUR,BE
17,304,2024-07-13,FR7612548029981234567890161,NaN,Espèces,Débit,Online,278.57,EUR,GB
19,295,2024-06-11,FR7610011000601234567890185,Particulier,Espèces,Crédit,Mobile,572.76,EUR,BE


### Combiner plusieurs conditions

- `&` signifie **ET** (toutes les conditions vraies)
- `|` signifie **OU** (au moins une vraie)
- ⚠️ Chaque condition doit être entre **parenthèses**.


In [21]:
# Espèces ET montant supérieur à 5 000
df[(df["transaction_type"] == "Espèces") & (df["amount"] > 5000)].head()

,transaction_id,date,account_id,counterparty,transaction_type,sens,channel,amount,currency,country
242,453,2024-02-17,FR7630004000031234567890143,DELTA LTD,Espèces,Débit,Online,10000.0,USD,FR


In [22]:
# Pays = US OU GB
df[df["country"].isin(["US", "GB"])].head()       # isin = "fait partie de cette liste"


,transaction_id,date,account_id,counterparty,transaction_type,sens,channel,amount,currency,country
0,358,2024-11-18,FR7612548029981234567890161,DELTA LTD,Virement,Crédit,Online,283.87,USD,GB
2,328,2024-07-10,FR7612548029981234567890161,ZETA BV,Carte,Crédit,Mobile,292.73,EUR,GB
3,14,2024-12-25,FR7610011000601234567890185,Particulier,Espèces,Débit,Mobile,888.46,EUR,US
7,174,2024-07-08,FR7620041010051234567890138,DELTA LTD,Virement,Crédit,Mobile,534.98,EUR,GB
9,467,2024-07-03,FR7612548029981234567890161,Particulier,Virement,Débit,Online,7491.24,EUR,US


In [23]:
# Montant compris entre 9 000 et 10 000
df[df["amount"].between(9000, 10000)].head()

,transaction_id,date,account_id,counterparty,transaction_type,sens,channel,amount,currency,country
41,377,2024-12-20,FR7610011000601234567890185,ALPHA SARL,Virement,Crédit,Online,9181.87,EUR,US
69,309,2024-08-25,FR7620041010051234567890138,DELTA LTD,Carte,Débit,Online,10000.00,EUR,DE
88,3,2024-09-27,FR7610011000601234567890185,BETA SA,Prélèvement,Débit,Online,9950.00,EUR,FR
89,59,2024-02-04,FR7620041010051234567890138,EPSILON SAS,Prélèvement,Débit,ATM,9800.00,GBP,LU
100,452,2024-04-27,FR7610011000601234567890185,GAMMA GMBH,Carte,Crédit,Agence,9800.00,USD,LU


In [24]:
# Le nom de contrepartie contient "SA" (recherche de texte)
df[df["counterparty"].str.contains("SA", na=False)].head()

,transaction_id,date,account_id,counterparty,transaction_type,sens,channel,amount,currency,country
5,404,2024-12-07,FR7612548029981234567890161,ALPHA SARL,Virement,Débit,Mobile,370.24,EUR,NaN
11,228,2024-10-06,FR7610011000601234567890185,EPSILON SAS,Prélèvement,Débit,ATM,154.88,EUR,GB
12,443,2024-05-06,FR7620041010051234567890138,BETA SA,Virement,Débit,ATM,273.03,EUR,US
14,257,2024-04-08,FR7610011000601234567890185,BETA SA,Virement,Crédit,Online,104.84,EUR,DE
15,473,2024-08-25,FR7620041010051234567890138,EPSILON SAS,Carte,Crédit,ATM,500.08,EUR,US


> ℹ️ `na=False` dit à pandas d'ignorer les valeurs manquantes pendant la recherche de texte,
> sinon elles provoqueraient une erreur.

Pour **compter** combien de lignes correspondent à un filtre, on enchaîne avec `.shape[0]` ou `len(...)` :


In [25]:
nb = len(df[df["amount"] > 10000])
print("Transactions > 10 000 :", nb)

Transactions > 10 000 : 12


### 🛠️ À vous de jouer #2

Combien de transactions sont des **Virement**s effectués via le canal **Online** ?
(Astuce : deux conditions reliées par `&`, puis `len(...)`.)


In [ ]:
# Votre code ici


**✅ Solution**

In [26]:
sel = df[(df["transaction_type"] == "Virement") & (df["channel"] == "Online")]
print("Virements Online :", len(sel))
sel.head()

Virements Online : 47


,transaction_id,date,account_id,counterparty,transaction_type,sens,channel,amount,currency,country
0,358,2024-11-18,FR7612548029981234567890161,DELTA LTD,Virement,Crédit,Online,283.87,USD,GB
9,467,2024-07-03,FR7612548029981234567890161,Particulier,Virement,Débit,Online,7491.24,EUR,US
14,257,2024-04-08,FR7610011000601234567890185,BETA SA,Virement,Crédit,Online,104.84,EUR,DE
30,362,2024-04-23,FR7630004000031234567890143,ETA TRADING,Virement,Débit,Online,312.35,EUR,DE
32,90,2024-07-06,FR7612548029981234567890161,BETA SA,Virement,Crédit,Online,972.99,EUR,DE


## 7. Trier

> 💡 **Équivalent Excel** : *Données → Trier* (croissant / décroissant).


In [27]:
# Tri par montant DÉCROISSANT (les plus gros en haut)
df.sort_values("amount", ascending=False).head()

,transaction_id,date,account_id,counterparty,transaction_type,sens,channel,amount,currency,country
364,444,2024-05-08,FR7630004000031234567890143,Particulier,Carte,Débit,Online,50000.0,EUR,GB
72,355,2024-08-10,FR7630004000031234567890143,NaN,Chèque,Crédit,ATM,50000.0,EUR,GB
93,207,2024-01-02,FR7630004000031234567890143,ALPHA SARL,Chèque,Débit,Mobile,50000.0,EUR,US
432,225,2024-05-02,FR7620041010051234567890138,DELTA LTD,Chèque,Débit,Online,50000.0,EUR,FR
323,64,2024-01-02,FR7620041010051234567890138,GAMMA GMBH,Carte,Débit,Mobile,50000.0,EUR,GB


In [28]:
# Tri sur deux colonnes : d'abord par compte, puis par date
df.sort_values(["account_id", "date"]).head()

,transaction_id,date,account_id,counterparty,transaction_type,sens,channel,amount,currency,country
418,497,2024-01-04,FR7610011000601234567890185,GAMMA GMBH,Prélèvement,Crédit,Mobile,300.50,EUR,LU
239,98,2024-01-09,FR7610011000601234567890185,ETA TRADING,Virement,Crédit,Mobile,742.66,USD,DE
435,152,2024-01-13,FR7610011000601234567890185,ZETA BV,Carte,Débit,Agence,297.89,EUR,US
359,135,2024-01-15,FR7610011000601234567890185,Particulier,Espèces,Débit,Agence,488.87,EUR,GB
444,93,2024-01-15,FR7610011000601234567890185,Particulier,Chèque,Crédit,Agence,1467.41,EUR,GB


## 8. Créer / calculer une nouvelle colonne

On crée une colonne en écrivant `df["nouveau_nom"] = ...`.

> 💡 **Équivalent Excel** : ajouter une colonne avec une formule qui se recopie sur toutes les lignes.


In [29]:
# Convertir tous les montants en EUR (taux fictifs, pour l'exemple)
taux = {"EUR": 1.0, "USD": 0.92, "GBP": 1.17}

df["taux_eur"]   = df["currency"].map(taux)      # map = associer chaque devise à son taux
df["amount_eur"] = (df["amount"] * df["taux_eur"]).round(2)

df[["amount", "currency", "taux_eur", "amount_eur"]].head()

,amount,currency,taux_eur,amount_eur
0,283.87,USD,0.92,261.16
1,1140.92,EUR,1.00,1140.92
2,292.73,EUR,1.00,292.73
3,888.46,EUR,1.00,888.46
4,187.45,EUR,1.00,187.45


In [ ]:
# Une colonne basée sur une condition : marquer les "gros" montants
df["gros_montant"] = df["amount_eur"] > 10000      # donne True / False
df[["amount_eur", "gros_montant"]].head()

### 🛠️ À vous de jouer #3

Créez une colonne **`montant_signe`** qui vaut le montant **négatif** quand le `sens` est `"Débit"`,
et **positif** quand c'est `"Crédit"`.

(Astuce : `np.where(condition, valeur_si_vrai, valeur_si_faux)` — l'équivalent du `SI()` d'Excel.)


In [ ]:
# Votre code ici


**✅ Solution**

In [30]:
df["montant_signe"] = np.where(df["sens"] == "Débit", -df["amount_eur"], df["amount_eur"])
df[["sens", "amount_eur", "montant_signe"]].head()

,sens,amount_eur,montant_signe
0,Crédit,261.16,261.16
1,Débit,1140.92,-1140.92
2,Crédit,292.73,292.73
3,Débit,888.46,-888.46
4,Crédit,187.45,187.45


## 9. Agréger : `sum`, `mean`, `count`, `min`, `max`

On applique un calcul à **toute une colonne**.

> 💡 **Équivalent Excel** : `=SOMME(...)`, `=MOYENNE(...)`, `=NB(...)`, `=MIN(...)`, `=MAX(...)`.


In [31]:
print("Total des montants (EUR) :", df["amount_eur"].sum().round(2))
print("Montant moyen (EUR)       :", df["amount_eur"].mean().round(2))
print("Montant médian (EUR)      :", df["amount_eur"].median().round(2))
print("Plus gros montant (EUR)   :", df["amount_eur"].max())
print("Nombre de transactions    :", df["amount_eur"].count())

Total des montants (EUR) : 1050360.59
Montant moyen (EUR)       : 2075.81
Montant médian (EUR)      : 681.0
Plus gros montant (EUR)   : 50000.0
Nombre de transactions    : 506


`value_counts()` compte combien de fois chaque valeur apparaît : parfait pour une colonne de catégories.

> 💡 **Équivalent Excel** : `NB.SI` répété pour chaque valeur, ou un tableau croisé en comptage.


In [32]:
df["transaction_type"].value_counts()

transaction_type
Virement       203
Carte          108
Prélèvement     91
Espèces         57
Chèque          47
Name: count, dtype: int64

## 10. Grouper : `groupby` ≈ `SOMME.SI` / `SUMIF`

`groupby` = « **pour chaque** catégorie, calcule… ». C'est l'outil clé pour les **synthèses**.

> 💡 **Équivalent Excel** : `SOMME.SI` / `SUMIF`, ou un **tableau croisé dynamique** simple.

Lecture du code ci-dessous : *« pour chaque `transaction_type`, fais la `sum` de `amount_eur` »*.


In [33]:
df.groupby("transaction_type")["amount_eur"].sum().round(2)

transaction_type
Carte          266754.59
Chèque         216793.64
Espèces         62855.66
Prélèvement    194201.35
Virement       309755.35
Name: amount_eur, dtype: float64

In [34]:
# On peut trier le résultat du plus gros au plus petit
df.groupby("transaction_type")["amount_eur"].sum().round(2).sort_values(ascending=False)

transaction_type
Virement       309755.35
Carte          266754.59
Chèque         216793.64
Prélèvement    194201.35
Espèces         62855.66
Name: amount_eur, dtype: float64

On peut grouper sur **plusieurs** colonnes, et demander **plusieurs** calculs à la fois avec `.agg(...)` :

In [35]:
# Pour chaque compte : total, moyenne et nombre d'opérations
df.groupby("account_id")["amount_eur"].agg(["sum", "mean", "count"]).round(2)

,sum,mean,count
account_id,,,
FR7610011000601234567890185,272677.33,1782.20,153
FR7612548029981234567890161,169484.78,1424.24,119
FR7620041010051234567890138,294746.57,2519.20,117
FR7630004000031234567890143,313451.91,2679.08,117


In [36]:
# Grouper sur deux niveaux : compte puis sens (Débit/Crédit)
df.groupby(["account_id", "sens"])["amount_eur"].sum().round(2)

account_id                   sens  
FR7610011000601234567890185  Crédit     85122.24
                             Débit     187555.09
FR7612548029981234567890161  Crédit     62845.15
                             Débit     106639.63
FR7620041010051234567890138  Crédit     72225.56
                             Débit     222521.01
FR7630004000031234567890143  Crédit    114563.74
                             Débit     198888.17
Name: amount_eur, dtype: float64

### 🛠️ À vous de jouer #4

Pour **chaque pays** (`country`), calculez le **nombre** de transactions et le **montant total** en EUR.
Quel pays a le total le plus élevé ?

(Astuce : `groupby("country")` puis `.agg(["count", "sum"])` sur `amount_eur`.)


In [ ]:
# Votre code ici


**✅ Solution**

In [37]:
resultat = df.groupby("country")["amount_eur"].agg(["count", "sum"]).round(2)
resultat.sort_values("sum", ascending=False)

,count,sum
country,,
US,91,295255.93
GB,87,227589.38
FR,77,165774.35
DE,94,141027.22
BE,77,114721.26
LU,73,102124.68


## 11. Tableaux croisés : `pivot_table`

Quand on veut une catégorie **en lignes** et une autre **en colonnes**, c'est exactement le
**tableau croisé dynamique** d'Excel.

> 💡 **Équivalent Excel** : *Insertion → Tableau croisé dynamique*.

Ici : `transaction_type` en lignes, `sens` en colonnes, et la **somme** des montants dans les cases.


In [38]:
pd.pivot_table(
    df,
    index="transaction_type",   # les lignes
    columns="sens",             # les colonnes
    values="amount_eur",        # la valeur à agréger
    aggfunc="sum",              # le calcul : "sum", "mean", "count"...
    fill_value=0,               # remplacer les cases vides par 0
).round(2)

sens,Crédit,Débit
transaction_type,,
Carte,69538.85,197215.74
Chèque,87745.21,129048.43
Espèces,22316.21,40539.45
Prélèvement,50130.76,144070.59
Virement,105025.66,204729.69


In [ ]:
# Même chose en COMPTANT le nombre d'opérations plutôt qu'en sommant
pd.pivot_table(df, index="channel", columns="sens",
               values="transaction_id", aggfunc="count", fill_value=0)

## 12. Rapprocher deux tables : `merge` ≈ `RECHERCHEV` / `VLOOKUP`

Très fréquent en audit : on a une table d'opérations, et **une autre table** de référence
(p. ex. les caractéristiques des comptes). On veut **ramener** les infos de la 2ᵉ table dans la 1ʳᵉ,
en s'appuyant sur une **colonne commune** (ici `account_id`).

> 💡 **Équivalent Excel** : `RECHERCHEV` / `VLOOKUP` (ou `RECHERCHEX` / `XLOOKUP`).

Créons d'abord une petite table de référence des comptes :


In [39]:
comptes_ref = pd.DataFrame({
    "account_id":   comptes,    # même variable que dans la section 2
    "titulaire":    ["Alpha Holding", "Beta Industries", "Gamma Trust", "Delta Family Office"],
    "segment":      ["Corporate", "Corporate", "Private", "Private"],
    "niveau_risque": ["Faible", "Moyen", "Élevé", "Moyen"],
})
comptes_ref

,account_id,titulaire,segment,niveau_risque
0,FR7610011000601234567890185,Alpha Holding,Corporate,Faible
1,FR7630004000031234567890143,Beta Industries,Corporate,Moyen
2,FR7612548029981234567890161,Gamma Trust,Private,Élevé
3,FR7620041010051234567890138,Delta Family Office,Private,Moyen


On **fusionne** maintenant `df` avec `comptes_ref` sur la colonne commune `account_id`.

- `on="account_id"` : la colonne de rapprochement (la « clé » du `VLOOKUP`).
- `how="left"` : on garde **toutes** les lignes de `df` (la table de gauche) — comme un `VLOOKUP`
  classique qui part de la table principale.


In [ ]:
df_enrichi = df.merge(comptes_ref, on="account_id", how="left")

df_enrichi[["transaction_id", "account_id", "titulaire", "segment", "niveau_risque", "amount_eur"]].head()

Maintenant qu'on a le `segment` et le `niveau_risque`, on peut faire des synthèses dessus —
c'est là que `merge` + `groupby` deviennent puissants :

In [ ]:
# Montant total par niveau de risque
df_enrichi.groupby("niveau_risque")["amount_eur"].sum().round(2).sort_values(ascending=False)

### 🛠️ À vous de jouer #5

À partir de `df_enrichi`, calculez le **montant total** et le **nombre d'opérations** par **`segment`**
(Corporate / Private).


In [ ]:
# Votre code ici


**✅ Solution**

In [ ]:
df_enrichi.groupby("segment")["amount_eur"].agg(["count", "sum"]).round(2)

## 13. Travailler avec les dates

La colonne `date` est déjà reconnue comme une vraie date (type *datetime*). On peut alors extraire
facilement l'année, le mois, le jour de la semaine… via l'accesseur `.dt`.

> 💡 **Équivalent Excel** : `ANNEE()`, `MOIS()`, `JOURSEM()`.


In [40]:
# Vérifier que la colonne est bien une date
df["date"].dtype

dtype('<M8[us]')

In [ ]:
df["annee"]      = df["date"].dt.year
df["mois"]       = df["date"].dt.month            # 1 à 12
df["jour_sem"]   = df["date"].dt.dayofweek        # 0 = lundi ... 6 = dimanche
df["nom_jour"]   = df["date"].dt.day_name()

df[["date", "annee", "mois", "jour_sem", "nom_jour"]].head()

**Évolution mensuelle** des montants — un classique de la revue analytique :

In [ ]:
evolution = df.groupby(df["date"].dt.to_period("M"))["amount_eur"].sum().round(2)
evolution

In [ ]:
# Un petit graphique en barres (optionnel mais parlant)
evolution.plot(kind="bar", figsize=(10, 3), title="Montant total par mois (EUR)");

## 14. Qualité des données : valeurs manquantes & doublons

Étape incontournable en audit : **fiabiliser** la donnée avant de l'analyser.

### Valeurs manquantes (`NaN` = *Not a Number*, une case vide)

> 💡 **Équivalent Excel** : repérer les cellules vides, `NB.VIDE`.


In [ ]:
# Combien de valeurs manquantes par colonne ?
df.isna().sum()

In [ ]:
# Voir les lignes où la contrepartie est manquante
df[df["counterparty"].isna()].head()

In [ ]:
# Deux façons de traiter :
df_sans_na = df.dropna(subset=["counterparty"])           # supprimer ces lignes
df_rempli  = df.fillna({"counterparty": "INCONNU",
                        "country": "INCONNU"})            # remplacer la case vide

print("Lignes après dropna :", len(df_sans_na))
print("Manquants après fillna :", df_rempli[["counterparty", "country"]].isna().sum().sum())

### Doublons

> 💡 **Équivalent Excel** : *Données → Supprimer les doublons*, ou une mise en forme conditionnelle
> « valeurs en double ».

`duplicated()` marque les lignes en double. On peut chercher les doublons sur **tout** ou sur un
**sous-ensemble de colonnes métier** (même date, même compte, même montant, même contrepartie…).


In [ ]:
# Doublons "métier" : même date, compte, contrepartie, type et montant
cles = ["date", "account_id", "counterparty", "transaction_type", "amount"]

doublons = df[df.duplicated(subset=cles, keep=False)]      # keep=False -> garde TOUTES les occurrences
print("Lignes impliquées dans un doublon :", len(doublons))
doublons.sort_values(cles).head(10)

In [ ]:
# Obtenir une table SANS doublons (on ne garde que la 1re occurrence)
df_dedup = df.drop_duplicates(subset=cles, keep="first")
print("Avant :", len(df), "-> Après :", len(df_dedup))

## 15. 🔎 Cas d'audit concrets

On combine maintenant tout ce qui précède pour quelques **tests analytiques** typiques.
⚠️ Ce sont des *exemples pédagogiques* simplifiés, pas une méthodologie de contrôle complète.


### 15.1 — Les 10 plus gros montants
Repérer les opérations les plus significatives.

In [ ]:
df.sort_values("amount_eur", ascending=False).head(10)[
    ["transaction_id", "date", "account_id", "counterparty", "amount_eur", "transaction_type"]
]

### 15.2 — Montants « ronds »
Des montants exactement ronds (multiples de 1 000) peuvent mériter une attention particulière.

In [ ]:
ronds = df[df["amount"] % 1000 == 0]
print("Opérations à montant rond :", len(ronds))
ronds[["transaction_id", "date", "counterparty", "amount", "transaction_type"]].head(10)

### 15.3 — Montants juste sous un seuil (*structuring*)
Opérations entre 9 000 et 9 999, c.-à-d. **juste en dessous** du seuil de déclaration de 10 000 :
un schéma classique de fractionnement à surveiller.

In [ ]:
sous_seuil = df[(df["amount"] >= 9000) & (df["amount"] < 10000)]
print("Opérations juste sous 10 000 :", len(sous_seuil))
sous_seuil[["transaction_id", "date", "account_id", "counterparty", "amount"]].sort_values("amount", ascending=False)

### 15.4 — Opérations le week-end
Activité un samedi/dimanche : parfois inhabituel selon le contexte.

In [ ]:
weekend = df[df["date"].dt.dayofweek >= 5]      # 5 = samedi, 6 = dimanche
print("Opérations le week-end :", len(weekend))
weekend[["transaction_id", "date", "nom_jour", "counterparty", "amount_eur"]].head(10)

### 15.5 — Espèces importantes
Cumul des opérations en espèces, par compte, au-dessus d'un seuil de matérialité.

In [ ]:
especes = df[(df["transaction_type"] == "Espèces") & (df["amount_eur"] > 3000)]
especes.groupby("account_id")["amount_eur"].agg(["count", "sum"]).round(2).sort_values("sum", ascending=False)

### 🛠️ À vous de jouer #6 (synthèse)

Construisez une table des opérations qui réunissent **toutes** ces conditions :
- type **`Virement`**,
- montant en EUR **supérieur à 8 000**,
- contrepartie située **hors zone** `LU`, `FR`, `DE`, `BE` (donc à l'étranger).

Triez le résultat par montant décroissant.

(Astuce : combinez plusieurs conditions avec `&`, et utilisez `~df["country"].isin([...])`
où le `~` signifie **NON / l'inverse**.)


In [ ]:
# Votre code ici


**✅ Solution**

In [ ]:
zone_locale = ["LU", "FR", "DE", "BE"]
suspect = df[
    (df["transaction_type"] == "Virement")
    & (df["amount_eur"] > 8000)
    & (~df["country"].isin(zone_locale))
]
suspect = suspect.sort_values("amount_eur", ascending=False)
print("Opérations correspondantes :", len(suspect))
suspect[["transaction_id", "date", "counterparty", "country", "amount_eur"]]

## 16. Exporter ses résultats

Une fois un échantillon ou une synthèse obtenue, on l'exporte vers Excel/CSV pour le partager
ou le documenter dans le dossier de travail.

> 💡 **Équivalent Excel** : *Enregistrer sous*.


In [ ]:
# On exporte la table de l'exercice 6 (les virements internationaux significatifs)
suspect.to_excel("operations_a_revoir.xlsx", index=False)
print("Fichier 'operations_a_revoir.xlsx' créé avec", len(suspect), "lignes.")

## 🎓 Récapitulatif : du tableur à pandas

| Besoin | Excel | pandas |
|---|---|---|
| Ouvrir un fichier | *Fichier → Ouvrir* | `pd.read_excel(...)` / `pd.read_csv(...)` |
| Regarder les données | faire défiler | `df.head()`, `df.info()`, `df.describe()` |
| Filtrer | filtres automatiques | `df[df["col"] > x]` |
| Trier | *Données → Trier* | `df.sort_values("col")` |
| Colonne calculée | formule recopiée | `df["nouvelle"] = ...` |
| Somme conditionnelle | `SOMME.SI` / `SUMIF` | `df.groupby("cat")["val"].sum()` |
| Tableau croisé | TCD | `pd.pivot_table(...)` |
| Recherche/jointure | `RECHERCHEV` / `VLOOKUP` | `df.merge(autre, on="clé")` |
| Supprimer doublons | *Supprimer les doublons* | `df.drop_duplicates()` |
| Cellules vides | repérage manuel | `df.isna()`, `df.fillna()`, `df.dropna()` |
| Enregistrer | *Enregistrer sous* | `df.to_excel(...)` / `df.to_csv(...)` |

**Pour aller plus loin** : modifiez les seuils des cas d'audit, changez les colonnes des `groupby`,
testez vos propres filtres. La meilleure façon d'apprendre pandas est de **casser puis réparer** 🙂.
